# TA 2: process the scans, with t0 determination

Template on 250709, scans `s40` and `s44` (200 fs steps). Needs the wavelength calibration
written by `ta_1_linear_itos`.

Steps: read, calibrate the wavelength axis, compare the scans, reject bad data, average,
subtract the pre-t0 spectrum, pick t0 points, fit the t0 curve and write it as a calibration
file, shift, save.

Each interactive tool is preceded by a **decisions** cell: paste there what its "Done" button
prints. The step after the tool uses that literal, not the widget, so "Run all" gives the same
result without clicking.

Needs `%matplotlib widget` (JupyterLab or VS Code, with `ipympl`).

In [ ]:
%matplotlib widget
import datetime
from pathlib import Path

import numpy as np
from numpy.polynomial import Polynomial

from edyx.calib import interp_calibration, read_calibration, write_calibration
from edyx.io import read_dat, stack_traces, write_dat
from edyx.plot import plot_maps
from edyx.tana import avg_scans, avg_spectra, shift_trace, subtract_pre_t0
from edyx.units import nm2ev
from pyspan import Overlay, ScanInspector, T0Picker
from pyspan.decisions import above_threshold, rejection_mask, rmsd_vs_median

## Parameters

In [ ]:
data_dir = Path("../../../edyx/examples/data")  # edyx example data; in the lab: Path(".")
fnames = sorted(data_dir.glob("s4?.dat"))
wl_file = Path("250709_wl_itos.toml")  # written by ta_1_linear_itos
t_pre = -0.7  # ps: delays t < t_pre are before time zero
date = datetime.date(2025, 7, 9)

out_dir = Path(".")
out_root = "s4044"
names = [f.name for f in fnames]
names

## Read and calibrate the wavelength axis

Axes of `stack`: (file, wl, delay). NaN become masked points, ignored by the averages.
The wavelength calibration is applied right after reading: every later step, t0 included, uses
the calibrated axis.

In [ ]:
t, wl_raw, stack = stack_traces([read_dat(f) for f in fnames])
stack = np.ma.masked_invalid(stack)
wl = interp_calibration(read_calibration(wl_file), wl_raw)
stack.shape

## Compare the scans

In [ ]:
Overlay.from_stack(t, wl, stack, labels=names, mode="kinetics").show()

## Reject data

RMSD of each scan against the median of the scans. With 2 files the median is the mean, so both
files get the same RMSD and the threshold cannot single out one file.

In [ ]:
rmsd = rmsd_vs_median(stack, axis=0)

In [ ]:
# Paste the output of "Done" here. Format: (file, wl, t) indices, None = all,
# (start, stop) = range, stop excluded. E.g. [(1, None, (250, None))]
rejections = []
threshold = None  # None: no threshold

In [ ]:
insp = ScanInspector(t, wl, stack, labels=names, rejections=rejections,
                     rmsd=rmsd, threshold=threshold)
insp.show()

In [ ]:
mask = rejection_mask(stack.shape, rejections)
if threshold is not None:
    mask |= rejection_mask(stack.shape, above_threshold(rmsd, threshold))
data = np.ma.masked_array(stack, mask=mask)  # the NaN masked at reading stay masked

## Average and subtract the pre-t0 spectrum

In [ ]:
avg = avg_scans(data)
neg_spec = avg_spectra(avg[..., t < t_pre])  # mean spectrum before time zero, saved below
avg_sub = subtract_pre_t0(t, avg, t_pre)    # avg - neg_spec

fig, axes = plot_maps(t, wl, [avg, avg_sub], titles=["average", "pre-t0 subtracted"],
                      vmin=-20, vmax=20)

In [ ]:
Overlay({"average": (t, wl, avg), "pre-t0 subtracted": (t, wl, avg_sub)}, mode="kinetics").show()

## Pick t0 points

Click on the kinetic trace at t0, one wavelength at a time ("Next after click" steps on). Then
paste the output of "Done" below and run again.

t0 from another measurement of the day instead: skip to the shift, with
`t0 = interp_calibration(read_calibration(t0_file), wl)`.

In [ ]:
# Paste the output of "Done" here. Rows: (wl, t0).
# Here: points picked on s20-s27 (20 fs steps), wavelengths converted to the calibrated axis.
t0_points = np.array([[ 3.2290e+02, -3.3100e-01],
     [ 3.2930e+02, -2.3500e-01],
     [ 3.3571e+02, -1.4600e-01],
     [ 3.4212e+02, -1.2800e-01],
     [ 3.5494e+02,  8.0000e-03],
     [ 3.4852e+02, -2.6000e-02],
     [ 3.8698e+02,  2.5800e-01],
     [ 3.9338e+02,  3.1400e-01],
     [ 3.9979e+02,  3.3000e-01],
     [ 4.1261e+02,  4.0800e-01],
     [ 4.1902e+02,  4.5800e-01],
     [ 4.2542e+02,  5.0500e-01],
     [ 4.3183e+02,  5.3700e-01],
     [ 4.3824e+02,  5.6300e-01],
     [ 4.4465e+02,  6.1000e-01],
     [ 4.5106e+02,  6.2500e-01],
     [ 4.5746e+02,  6.4700e-01],
     [ 4.6388e+02,  6.5300e-01],
     [ 4.7028e+02,  6.6500e-01],
     [ 4.7669e+02,  6.8000e-01],
     [ 4.8310e+02,  6.7800e-01],
     [ 4.8951e+02,  7.0600e-01],
     [ 4.9592e+02,  7.1100e-01],
     [ 5.0232e+02,  7.0100e-01],
     [ 5.0873e+02,  7.1100e-01],
     [ 5.1514e+02,  7.3500e-01],
     [ 5.2155e+02,  7.5500e-01],
     [ 5.2796e+02,  7.6300e-01],
     [ 5.3437e+02,  7.5900e-01],
     [ 5.4077e+02,  7.4900e-01],
     [ 5.4718e+02,  7.9500e-01],
     [ 5.5359e+02,  8.3100e-01],
     [ 5.6000e+02,  8.5500e-01],
     [ 5.6641e+02,  8.5500e-01],
     [ 5.7281e+02,  8.5300e-01],
     [ 5.7922e+02,  8.5900e-01],
     [ 5.8563e+02,  8.6700e-01],
     [ 5.9204e+02,  8.6900e-01],
     [ 5.9845e+02,  8.6900e-01],
     [ 6.0485e+02,  8.7100e-01],
     [ 6.1127e+02,  8.8100e-01],
     [ 6.1767e+02,  8.7300e-01],
     [ 6.2408e+02,  8.7300e-01],
     [ 6.3049e+02,  8.7100e-01],
     [ 6.3690e+02,  8.8000e-01],
     [ 6.4331e+02,  8.9400e-01],
     [ 6.4971e+02,  9.1800e-01],
     [ 6.5612e+02,  9.1800e-01],
     [ 6.6253e+02,  9.2800e-01],
     [ 6.6894e+02,  9.3000e-01],
     [ 6.7535e+02,  9.3400e-01],
     [ 6.8175e+02,  9.3000e-01],
     [ 6.8736e+02,  9.4800e-01]])

In [ ]:
picker = T0Picker(t, wl, avg_sub, points=t0_points, tlim=(-2, 2), clim=(-20, 20))
picker.show()

## Fit the t0 curve

Plain numpy: the model is your choice. Points are `(wl, t0)`. The calibration file holds the
sampled curve; the model and coefficients are kept for information.

In [ ]:
pol = Polynomial.fit(nm2ev(t0_points[:, 0]), t0_points[:, 1], 3).convert()  # choice: cubic in eV
t0 = pol(nm2ev(wl))

fig, axes = plot_maps(t, wl, [avg_sub], titles=["t0 points and fit"], vmin=-20, vmax=20)
axes[0, 0].plot(t0_points[:, 1], t0_points[:, 0], "k.", ms=3)
axes[0, 0].plot(t0, wl, color="k", lw=0.5)
axes[0, 0].set_xlim(-2, 2)

In [ ]:
out_dir.mkdir(exist_ok=True)
t0_file = out_dir / f"{date:%y%m%d}_t0_{out_root}.toml"
write_calibration(t0_file, "t0", "wl", wl, t0, units=["nm", "ps"], date=date, source=out_root,
                  made_by="ta_2_processing.ipynb",
                  fit={"model": "polynomial in eV, low order first", "coef": pol.coef,
                       "points": t0_points})

## Shift

`shift_trace` moves the signal at `t0[i]` to delay 0, and keeps only the delays valid for every
wavelength.

In [ ]:
t_c, shifted = shift_trace(t, avg_sub, t0)

fig, axes = plot_maps(t_c, wl, [shifted], titles=["t0 corrected"], vmin=-20, vmax=20)
axes[0, 0].axvline(0, color="k", lw=0.5)
axes[0, 0].set_xlim(-2, 2)

## Save

In [ ]:
header = (f"scans: {names}\nt_pre: {t_pre} ps\n"
          f"wl calibration: {wl_file.name}\nt0 curve: {t0_file.name}")
write_dat(out_dir / f"{date:%y%m%d}_{out_root}.dat", t_c, wl, shifted, header=header)
np.savetxt(out_dir / f"{date:%y%m%d}_{out_root}_negdelay.dat", np.column_stack([wl, neg_spec.filled(np.nan)]),
           fmt="%.6e", header="wavelength / nm, dA / mOD")

fig, axes = plot_maps(t_c, wl, [shifted], vmin=-15, vmax=15)
fig.savefig(out_dir / f"{date:%y%m%d}_{out_root}.png", dpi=300)